## `ping` — First Live Reachability Test

The `ping` command is a network diagnostic tool used to test the connectivity between a **local host** and a **remote host**. It sends Internet Control Message Protocol (ICMP) **echo request packets** to the **target host** and waits for the ICMP echo reply.

> `ping hostname` secretly does **two things** in sequence — (1) resolve the hostname to an IP, then (2) send ICMP to that IP.

- Test network connectivity
- Test network interface card
- Test DNS name resolution issues

**Support context:**
> `ip addr` / `ip route` / `ip link` tell you the *local* box is configured correctly — interface up, valid IP, routes present. **None of that proves anything actually answers on the other end**.

> `ping` is the first command that leaves your machine and asks a real question: "is there anything alive at this address at all?" It's the **cheapest, fastest binary answer** in your whole toolkit — **always reach for it before DNS tools, before `curl`, before anything app-layer**.

**COMMAND:** `ping`

**PURPOSE (support):** Splits a huge class of tickets in half in one command.
- **"Can't reach the app"** → ping the target. No reply at all → the problem is likely network/routing (below the app layer) investigate it.
- **Replies fine but the app still fails** → the problem is above the network layer (closed port, crashed process, app-level firewall rule) — network connectivity itself is proven fine, so stop looking there. Network is cleared, move up-stack to port/process/firewall (`nc -zv`, `curl`, checking if the process is even running). That's the correct next move, you just can't say "the app is up" from ping alone — only "the network path to it is fine."One command, and you already know **which half of the stack to investigate next**.

✅ A successful ping only proves the **host answered ICMP** — nothing about the specific service/port the app runs on. The host being alive and the app process being alive/listening are two separate facts.

> 💡 **Why**: many hosts respond to ICMP via the OS kernel itself, completely independent of whether the app process on top is running, crashed, or hung. A dead app on a perfectly healthy host still pings fine.

> 🔁 **Support rule**: ping proves reachability, not service health.

**SYNTAX:** `ping [options] destination`

**FLAGS:**

|Flag|Meaning|
|---|---|
|`-c N`|send exactly N packets then stop — **use this every time**, see GOTCHA|
|`-i seconds`|interval between packets (default 1s)|
|`-W seconds`|timeout to wait for each reply|
|`-s bytes`|packet payload size — useful for MTU/fragmentation testing|
|`-4` / `-6`|force IPv4 / IPv6|

**Developer bridge:** think of `ping` like a network-layer liveness probe — the same idea as a `/healthz` endpoint, except instead of an HTTP GET it sends an **ICMP echo request** and waits for an **ICMP echo reply**.

> Crucially, like a liveness probe that only checks "process is running" and says nothing about whether the app is actually serving correct data, `ping` only proves the host answered *something* — it says nothing about whether the service/port you actually care about is up.

**EXAMPLE:**

Ping an IP address. This host can be **any** network device such as a computer, router or printer whether it's on LAN or over the internet.

We send 4 data packets, than wait for a response. Then the sever will send the data packets back to us as the reply. These replies are called **echo reply requests**. These replies tell you about what's happening with the server we ping.

In [ ]:
$ ping -c 4 8.8.8.8
PING 8.8.8.8 (8.8.8.8) 56(84) bytes of data.
64 bytes from 8.8.8.8: icmp_seq=1 ttl=115 time=14.2 ms
64 bytes from 8.8.8.8: icmp_seq=2 ttl=115 time=13.8 ms
64 bytes from 8.8.8.8: icmp_seq=3 ttl=115 time=14.5 ms
64 bytes from 8.8.8.8: icmp_seq=4 ttl=115 time=13.9 ms

--- 8.8.8.8 ping statistics ---
4 packets transmitted, 4 received, 0% packet loss, time 3005ms
rtt min/avg/max/mdev = 13.8/14.1/14.5/0.3 ms

**READ THE OUTPUT:**

- **`icmp_seq=1,2,3,4`** — sequence numbers. Watch for **gaps** (1, 2, 4 — missing 3) across a longer run; a gap is a dropped packet even if the summary line's percentage looks small.
- **`ttl=115`** — Time To Live, decremented by one at every router hop the packet crosses. Not something you'll typically act on directly, but a wildly unexpected TTL (way lower than you'd expect for a nearby host) can hint at a routing loop or an unexpected extra hop.
- **`time=14.2 ms`** — round-trip time for that one packet. This is your latency number.
- **`0% packet loss`** — the single most important stat in the output. Healthy = `0%`. Any nonzero loss on a link that should be stable is worth escalating, even if most packets got through.
- **`rtt min/avg/max/mdev`** — `mdev` (mean deviation) is the one people skip and shouldn't: high `mdev` means **jittery, unstable** timing even when `avg` looks fine. A connection bouncing between 5ms and 200ms can average to a perfectly reasonable-looking 40ms — `mdev` is what exposes that instability.

If there is no reply than that means the server did not reply back, and it could mean that there is no network connectivity between our computer and the server. This could be for several different reasons:
- After the message has been sent we recieved a message back with `request time out` that could mean the **sever is powered down** or the server is up and running but it's using a **firewall that is blocking all ping requests**.
- **Not all** data packets reply back - **packet loss** `loss 50%` could happen for several reasons:
  - Network congestion: there is a lot of traffic, the network cannot handle all that traffic. The data packets may be dropped
  - Faulty hardware: bad cables or bad wiring
  - Bad network card or bad connections, or bad modem
- **Get message** `Destination host unreachable`: this means
  - a route to the destination cannot be found. It could mean that a router doesn't have any informaiton on how to route data to the destination
  - the remote server is down or disconnected from the network
  - your computer is not connected to the network

#### Scenario
Someone says they don't have an internet, because they cannot access any web pages with their computer, which is a pretty common issue.

**Diagnosis path**:
- **Ping test**. Sit at their computer and use `ping` utility to see if they are not connected to the internet. Even before checking their modem or router or any cable connection the **first thing I do** is do a simple `ping` test for example to `google.com`. Pinging the **domain name** and not the IP address since this way we also test **name resolution** issues related to DNS and also besides that, I don't have `google.com` IP address memorized. So, by pinging `google.com` and if the ping is successful, then I know that we are connected to the internet because I got a successful reply from `google.com`. With the successful reply I automatically know that all of the hardware (the cables, the network cards, the router, the modem) are all working well, including ISP. Now I know the issue is **related to sofware** and not hardware. The issue is most likely with their internet browser or it's a firewall issue. By doing the ping test saved a lot of time becaused I bypassed the hassle of looking around for the modem, the modem, of looking at LEDs, or checking the cables, or looking behind the computer to see if a network cable is plugged into the network card. With the simple **ping test**.
- **Check hardware**. The only time you need you hardware if the ping test **failed**.

**Testing the Network card - Loopback Test**. A loopback test will send out signals back to your computer for testing purposes.

Loopback:
```shell
ping 127.0.0.1
```
If the ping was successful than you can be assured that your network card is working properly.

If it failed - there is a problem with your **network card**.

#### Test DNS name resolution issues

In [ ]:
dennie@X390:~/linux-tut$ ping -c 4 yahoo.com
PING yahoo.com (74.6.231.21) 56(84) bytes of data.
64 bytes from media-router-fp74.prod.media.vip.ne1.yahoo.com (74.6.231.21): icmp_seq=1 ttl=49 time=180 ms
64 bytes from media-router-fp74.prod.media.vip.ne1.yahoo.com (74.6.231.21): icmp_seq=2 ttl=49 time=207 ms
64 bytes from media-router-fp74.prod.media.vip.ne1.yahoo.com (74.6.231.21): icmp_seq=3 ttl=49 time=188 ms
64 bytes from media-router-fp74.prod.media.vip.ne1.yahoo.com (74.6.231.21): icmp_seq=4 ttl=49 time=159 ms

--- yahoo.com ping statistics ---
4 packets transmitted, 4 received, 0% packet loss, time 2998ms
rtt min/avg/max/mdev = 159.446/183.893/207.284/17.158 ms

The message `Ping request could not find host yahoo.com. Please check the name and try again.` could mean a DNS issue because our computer could not resolve the domain name to IP address. The next thing to do is to ping the IP address instead and that is if you already knew what the IP address was already. If you don't remember the IP you can use an **easy IP address to remember** which is Google's DNS IP address which is 8.8.8.8. If the ping was successful this time then then this comfirms it's a DNS issue because the computer can ping the IP addresses but **it cannot ping domain names**.

**Solution**:
- Flushing your DNS by `ipconfig /flushdns`
- Chech you DNS settings on your network card configuration
- It could also be a problem with DNS servers iteslf
- It could be a problem with ISP
- Also could be checked with ping and subcommand `switches`. Switches are used to alter the parameters of the ping utility and you can view a full list of switches by `ping /?`

**TRY NOW:**

Run these in your WSL terminal and paste the real output into a cell below. **Always pass `-c`** — see the GOTCHA before you forget and run a bare `ping` in an interview or on a shared box.

```
ping -c 4 8.8.8.8
ping -c 4 google.com
ping -c 4 <your default gateway from the ip route lesson>
```

The second one matters beyond "does it work": if `ping 8.8.8.8` (a raw IP) succeeds but `ping google.com` (a hostname) fails or hangs, that's not a network problem anymore — that's a **DNS** problem. Same destination, completely different fix path. Keep the gateway output handy too — we'll compare it against the internet-facing pings once you paste both.

In [ ]:
dennie@X390:~/linux-tut$ ping -c 4 8.8.8.8
PING 8.8.8.8 (8.8.8.8) 56(84) bytes of data.
64 bytes from 8.8.8.8: icmp_seq=1 ttl=114 time=197 ms
64 bytes from 8.8.8.8: icmp_seq=2 ttl=114 time=77.2 ms
64 bytes from 8.8.8.8: icmp_seq=3 ttl=114 time=75.0 ms
64 bytes from 8.8.8.8: icmp_seq=4 ttl=114 time=65.7 ms

--- 8.8.8.8 ping statistics ---
4 packets transmitted, 4 received, 0% packet loss, time 3116ms
rtt min/avg/max/mdev = 65.664/103.779/197.167/54.092 ms

`ping -c 4 8.8.8.8` is in and worth a closer look — `0% packet loss`, but the timing tells a story:

- `icmp_seq=1` came back at **197ms**, then packets 2–4 settled to **65–77ms**. That's a ~3x drop after the first packet.
- **`mdev = 54.092`** — high, and it's almost entirely explained by that one slow first packet. If all four had bounced around unpredictably (say 60, 190, 70, 180), that'd be genuine ongoing jitter — a flaky link worth escalating. One slow-then-stable pattern like this is usually just a **cold path**: first packet pays a one-time cost (route/ARP-cache warm-up somewhere along the path, or your own or an upstream device's first-hit latency), then subsequent packets ride the now-warm path.
- **Verdict:** healthy connection. If this were a real ticket, you'd want more than 4 packets before ruling out jitter entirely (`ping -c 20 8.8.8.8` gives a much more trustworthy `mdev`) — 4 packets is really only enough to catch total failure, not to characterize stability with confidence.

In [ ]:
dennie@X390:~/linux-tut$ ping -c 4 google.com
PING google.com (142.251.98.102) 56(84) bytes of data.
64 bytes from nt-in-f102.1e100.net (142.251.98.102): icmp_seq=1 ttl=111 time=126 ms
64 bytes from nt-in-f102.1e100.net (142.251.98.102): icmp_seq=2 ttl=111 time=75.0 ms
64 bytes from nt-in-f102.1e100.net (142.251.98.102): icmp_seq=3 ttl=111 time=91.1 ms
64 bytes from nt-in-f102.1e100.net (142.251.98.102): icmp_seq=4 ttl=111 time=138 ms

--- google.com ping statistics ---
4 packets transmitted, 4 received, 0% packet loss, time 3112ms
rtt min/avg/max/mdev = 75.044/107.497/137.712/25.409 ms

In [ ]:
dennie@X390:~/linux-tut$ ip route
default via 172.30.240.1 dev eth0 proto kernel
172.30.240.0/20 dev eth0 proto kernel scope link src 172.30.240.133

dennie@X390:~/linux-tut$ ping -c 4 172.30.240.1
PING 172.30.240.1 (172.30.240.1) 56(84) bytes of data.

--- 172.30.240.1 ping statistics ---
4 packets transmitted, 0 received, 100% packet loss, time 3193ms

Three results, and the third one is the interesting one:

- **`ping -c 4 google.com`** — resolved to `142.251.98.102`, `0% packet loss`, `mdev = 25.4`. Healthy, and it answers the DNS question from the TRY NOW instructions: since `8.8.8.8` and `google.com` both worked, DNS resolution on this box is fine — no fork needed here.
- **`ping -c 4 172.30.240.1`** (your default gateway) — **100% packet loss, zero replies.** At first glance this looks like the worst result of the three: your own gateway not responding at all. But look at what you just proved two commands earlier — traffic to `8.8.8.8` and `google.com` both went *through* this exact gateway and came back fine. **The gateway is clearly routing traffic correctly; it's just not answering ICMP echo requests addressed to itself.**

> This is exactly the GOTCHA from the lesson, caught live: **ping failure ≠ device down.** WSL2's virtual gateway is a common real-world example — it silently drops ICMP addressed directly to it while still forwarding everything through it perfectly. If this were a real ticket ("I can't ping the gateway, is our router down?") the correct read is: *check whether traffic actually passes through it* (which you already did) before ever concluding the gateway itself is the problem. Pinging the gateway directly is a reasonable first move, but a failure there is a lead to investigate, not a verdict.

**TICKET SCENARIO:** "Users report our API at `api.internal.example.com` is completely unreachable — timeouts on every request." Before touching the app, load balancer, or firewall rules: `ping api.internal.example.com`.

- **No reply at all** → likely a network/routing/DNS issue, below the application. Check `ip route`, then whether the hostname even resolves.
- **Replies fine, 0% loss** → the network path is proven healthy. The problem has to be at or above the app layer — the process crashed, the port isn't listening, a firewall rule blocks that specific port while allowing ICMP, etc. You just eliminated an entire category of suspects in one command.

**DIAGNOSIS PATH:**

1. `ping <default gateway>` — can you even leave your own machine? If this fails, nothing past it matters yet — fix local routing first (back to the `ip route` lesson).
2. `ping <target IP>` — bypass DNS entirely, test pure IP reachability to the actual destination.
3. `ping <target hostname>` — if step 2 worked but this doesn't (or hangs), the network path is fine and DNS resolution is the actual fault. Two very different fixes from what looks like the same symptom to the end user.
4. Read `0% packet loss` and `mdev`, not just "did I get any replies" — a flaky link with 20% loss is a real, ongoing problem even though it "technically" responded.

**GOTCHA:** Run `ping` with **no `-c` flag** and it runs **forever** until you `Ctrl+C` it — a classic footgun when you're pasting a command from a ticket or runbook into a live terminal, especially over a slow/remote session. Always default to `-c 4` out of habit.
> Second, bigger gotcha: **ping success is not the same as service up.** Many hosts and firewalls deliberately block ICMP (Internet Control Message Protocol) entirely (common hardening practice) while the actual service (port 443, port 22, whatever) works completely fine — so a failed ping doesn't always mean the host is down, just that ICMP specifically is blocked. Don't close a ticket as "host unreachable" off a failed ping alone; confirm with a port-level check (`curl`, `nc`, `ss` — coming up later in this pillar) before concluding the host itself is down.